# GenSivana Level 3 — V3 Clean Self-Select

Notebook autonome dérivé du run public **0.87076**. Il compare la recette V2
gagnante à une V3 haute résolution sur des groupes perceptuels identiques,
sélectionne la meilleure sans consulter le leaderboard, puis réentraîne le
candidat retenu sur toutes les données étiquetées.

Contraintes : un seul GPU T4, moins de sept heures, aucune dépendance envers un
autre notebook, score mesuré après le véritable round-trip polygone.


In [1]:
%pip install -q imagehash==4.3.2 timm==1.0.26
%pip install -q albumentations==2.0.8 opencv-python-headless


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import copy  # noqa: E402
import gc  # noqa: E402
import json  # noqa: E402
import math  # noqa: E402
import platform  # noqa: E402
import random  # noqa: E402
import time  # noqa: E402
import warnings  # noqa: E402
import zipfile  # noqa: E402
from pathlib import Path  # noqa: E402
import albumentations as A  # noqa: E402
import cv2  # noqa: E402
import imagehash  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import sklearn  # noqa: E402
import timm  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from PIL import Image, ImageFile  # noqa: E402
from sklearn.metrics import confusion_matrix  # noqa: E402
from sklearn.model_selection import StratifiedGroupKFold  # noqa: E402
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler  # noqa: E402

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")
SEED = 20260727
BASE_SIZE, V3_SIZE, EVAL_SIZE = 512, 640, 640
DEV_EPOCHS, FINAL_EPOCHS = 6, 8
MIN_V3_GAIN, MIN_TTA_GAIN, MIN_RETRIEVAL_GAIN = 0.002, 0.001, 0.0005
MAX_COMPARE_HOURS, MAX_REFIT_START_HOURS = 3.75, 4.25
MAX_START_EPOCH_HOURS, MAX_TOTAL_HOURS = 5.50, 6.75
RUN_TAG = "level3_v3_self_select_s20260728"
POLYGON_SEPARATOR = chr(59)
WINNER_SOURCE_SHA256 = (
    "bd5dae5fff873e371e9b7f5bdd311d98e" "c9c7240dd65bc2628fdf93125b7d355"
)
OUT = Path("/kaggle/working")
if not OUT.exists():
    OUT = Path("./working_level3_v3")
OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda" or torch.cuda.device_count() != 1:
    raise RuntimeError("Le run final exige exactement un GPU CUDA visible")
START_TIME = time.monotonic()
WARNINGS = []
PHASE_TIMES = {}


def elapsed_hours():
    """Return the elapsed runtime in hours.

    Returns:
        Hours elapsed since the notebook initialization."""
    return (time.monotonic() - START_TIME) / 3600


def seed_all(seed=SEED):
    """Seed every random number generator used by the notebook.

    Args:
        seed: Deterministic seed shared by Python, NumPy, PyTorch, and CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    """Seed one data-loading worker deterministically.

    Args:
        worker_id: Zero-based identifier assigned by the data loader."""
    value = (SEED + worker_id) % (2**32)
    random.seed(value)
    np.random.seed(value)


seed_all()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("device", DEVICE, torch.cuda.get_device_name(0))


device cuda Tesla T4


## 1. Données et codec officiel

Les polygones sont décodés à la taille originale. La sérialisation conserve
toutes les composantes externes éligibles, conformément au notebook gagnant.


In [3]:
def find_root():
    """Locate the Level 3 competition dataset.

    Returns:
        Directory containing train, validation, test, and submission files.

    Raises:
        FileNotFoundError: If no supported dataset layout is available."""
    candidates = [
        Path("/kaggle/input/competitions/gensivana-real-or-fake-level-3-localization"),
        Path("/kaggle/input/gensivana-real-or-fake-level-3-localization"),
        Path("."),
    ]
    for root in candidates:
        required = [
            root / "train/segmentation.csv",
            root / "val/segmentation.csv",
            root / "test/images",
            root / "sample_submission.csv",
        ]
        if all(path.exists() for path in required):
            return root
    raise FileNotFoundError("Dataset Level 3 introuvable")


def is_blank(value):
    """Check whether a polygon value represents an empty mask.

    Args:
        value: Raw polygon value read from a CSV file.

    Returns:
        True when the value is missing or contains only whitespace."""
    return not isinstance(value, str) or not value.strip()


def parse_polygons(value):
    """Parse normalized polygon components from one value.

    Args:
        value: Encoded polygon string or an empty value.

    Returns:
        List of finite float arrays shaped as ``(points, 2)``."""
    if is_blank(value):
        return []
    result = []
    for part in str(value).split(POLYGON_SEPARATOR):
        tokens = part.strip().split()
        if len(tokens) >= 6 and len(tokens) % 2 == 0:
            points = np.asarray(tokens, np.float32).reshape(-1, 2)
            if np.isfinite(points).all():
                result.append(np.clip(points, 0, 1))
    return result


def polygon_to_mask(value, height, width):
    """Rasterize normalized polygons into a binary mask.

    Args:
        value: Encoded polygon components.
        height: Output mask height in pixels.
        width: Output mask width in pixels.

    Returns:
        Unsigned binary mask with shape ``(height, width)``."""
    mask = np.zeros((height, width), np.uint8)
    for points in parse_polygons(value):
        scaled = points.copy()
        scaled[:, 0] *= width
        scaled[:, 1] *= height
        scaled = np.rint(scaled).astype(np.int32)
        scaled[:, 0] = np.clip(scaled[:, 0], 0, width - 1)
        scaled[:, 1] = np.clip(scaled[:, 1], 0, height - 1)
        cv2.fillPoly(mask, [scaled], 1)
    return mask


def mask_to_polygon(mask, epsilon_fraction=0.0, min_area=3):
    """Serialize every eligible external mask component.

    Args:
        mask: Two-dimensional binary mask.
        epsilon_fraction: Contour simplification relative to its perimeter.
        min_area: Minimum contour area retained in pixels.

    Returns:
        Normalized polygon string or one space for an empty prediction."""
    binary = np.asarray(mask, np.uint8)
    if not binary.any():
        return " "
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = binary.shape
    encoded = []
    for contour in sorted(contours, key=cv2.contourArea, reverse=True):
        if cv2.contourArea(contour) < min_area:
            continue
        perimeter = cv2.arcLength(contour, True)
        points = cv2.approxPolyDP(contour, epsilon_fraction * perimeter, True)
        if len(points) < 3:
            points = contour
        points = points.reshape(-1, 2).astype(np.float32)
        points[:, 0] /= width
        points[:, 1] /= height
        encoded.append(" ".join(f"{x:.6f}" for x in np.clip(points, 0, 1).ravel()))
    if encoded:
        return POLYGON_SEPARATOR.join(encoded)
    return " "


def official_iou(truth, prediction):
    """Compute IoU with the competition empty-mask convention.

    Args:
        truth: Ground-truth binary mask.
        prediction: Predicted binary mask with the same shape.

    Returns:
        Intersection over union in the inclusive range from zero to one."""
    truth, prediction = np.asarray(truth, bool), np.asarray(prediction, bool)
    union = np.logical_or(truth, prediction).sum()
    return 1.0 if union == 0 else float(np.logical_and(truth, prediction).sum() / union)


def read_split(root, name, labelled=True):
    """Load and validate one competition split.

    Args:
        root: Competition dataset directory.
        name: Split name such as train, val, or test.
        labelled: Whether a segmentation CSV is available for the split.

    Returns:
        Data frame containing paths, polygons, dimensions, and mask labels.

    Raises:
        FileNotFoundError: If at least one referenced image is absent."""
    image_dir = root / name / "images"
    frame = (
        pd.read_csv(root / name / "segmentation.csv", keep_default_na=False)
        if labelled
        else pd.read_csv(root / "sample_submission.csv", keep_default_na=False)[
            ["image_id"]
        ]
    )
    if not labelled:
        frame["polygon"] = " "
    frame["path"] = frame.image_id.map(image_dir.__truediv__)
    if not frame.path.map(Path.exists).all():
        raise FileNotFoundError(f"Images manquantes dans {name}")
    sizes = []
    for path in frame.path:
        with Image.open(path) as image:
            sizes.append((image.height, image.width))
    frame[["height", "width"]] = pd.DataFrame(sizes, index=frame.index)
    frame["has_mask"] = ~frame.polygon.map(is_blank)
    frame["split"] = name
    return frame.reset_index(drop=True)


ROOT = find_root()
train_df = read_split(ROOT, "train")
val_df = read_split(ROOT, "val")
test_df = read_split(ROOT, "test", False)
sample_submission = pd.read_csv(ROOT / "sample_submission.csv", keep_default_na=False)
test_mask = np.zeros((41, 53), np.uint8)
test_mask[2:12, 3:18] = 1
test_mask[22:38, 31:49] = 1
encoded = mask_to_polygon(test_mask)
assert official_iou(test_mask, polygon_to_mask(encoded, 41, 53)) > 0.90
assert mask_to_polygon(np.zeros((8, 9), np.uint8)) == " "
assert POLYGON_SEPARATOR in encoded
print(
    ROOT,
    len(train_df),
    len(val_df),
    len(test_df),
    "native sizes",
    train_df[["height", "width"]].drop_duplicates().values.tolist(),
)


/kaggle/input/competitions/gensivana-real-or-fake-level-3-localization 7857 978 993 native sizes [[640, 640]]


## 2. Groupes perceptuels et splits anti-fuite


In [4]:
def compute_hashes(frame, cache_name):
    """Compute or restore perceptual hashes for a data frame.

    Args:
        frame: Data frame containing image paths.
        cache_name: File name used for the NumPy cache.

    Returns:
        Hexadecimal perceptual hash for every row in input order."""
    cache = OUT / cache_name
    if cache.exists():
        values = np.load(cache, allow_pickle=True).tolist()
        if len(values) == len(frame):
            return values
    values = []
    for path in frame.path:
        with Image.open(path) as image:
            values.append(str(imagehash.phash(image.convert("RGB"))))
    np.save(cache, np.asarray(values, object))
    return values


def perceptual_groups(hashes, maximum_distance=4):
    """Group perceptually similar images with union-find.

    Args:
        hashes: Sequence of hexadecimal 64-bit perceptual hashes.
        maximum_distance: Largest Hamming distance allowed within a group.

    Returns:
        Integer group identifier for every input hash."""
    parent = np.arange(len(hashes))
    integers = [int(x, 16) for x in hashes]
    buckets = {}

    def find(i):
        """Find and compress one union-find root.

        Args:
            i: Index whose representative is requested.

        Returns:
            Representative index of the connected component."""
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        """Merge two union-find components.

        Args:
            a: Index from the first component.
            b: Index from the second component."""
        a, b = find(a), find(b)
        if a != b:
            parent[b] = a

    for i, value in enumerate(integers):
        keys = [(value >> shift) & 255 for shift in (56, 48, 40, 32, 24, 16, 8, 0)]
        candidates = set()
        for pos, key in enumerate(keys):
            candidates.update(buckets.get((pos, key), []))
        for other in candidates:
            if (value ^ integers[other]).bit_count() <= maximum_distance:
                union(i, other)
        for pos, key in enumerate(keys):
            buckets.setdefault((pos, key), []).append(i)
    return np.asarray([find(i) for i in range(len(hashes))])


train_hashes = compute_hashes(train_df, f"{RUN_TAG}_train_phash.npy")
groups = perceptual_groups(train_hashes)
outer = StratifiedGroupKFold(7, shuffle=True, random_state=SEED)
development_index, calib_index = next(
    outer.split(train_df, train_df.has_mask.astype(int), groups)
)
development = train_df.iloc[development_index]
development_groups = groups[development_index]
inner = StratifiedGroupKFold(6, shuffle=True, random_state=SEED + 1)
fit_local, selection_local = next(
    inner.split(development, development.has_mask.astype(int), development_groups)
)
fit_index, selection_index = (
    development_index[fit_local],
    development_index[selection_local],
)
fit_df, selection_df, calib_df = [
    train_df.iloc[x].reset_index(drop=True)
    for x in (fit_index, selection_index, calib_index)
]
sets = [set(groups[x]) for x in (fit_index, selection_index, calib_index)]
assert (
    sets[0].isdisjoint(sets[1])
    and sets[0].isdisjoint(sets[2])
    and sets[1].isdisjoint(sets[2])
)
print(
    "fit/selection/calib",
    len(fit_df),
    len(selection_df),
    len(calib_df),
    "groups",
    len(set(groups)),
)


fit/selection/calib 5596 1127 1134 groups 6898


## 3. Dataset, V2 gagnante et V3 haute résolution


In [5]:
MEAN = np.asarray([0.485, 0.456, 0.406], np.float32)
STD = np.asarray([0.229, 0.224, 0.225], np.float32)


def transforms(size, training):
    """Build aligned image and mask transformations.

    Args:
        size: Square output resolution in pixels.
        training: Whether stochastic training augmentation is enabled.

    Returns:
        Albumentations composition applied jointly to image and mask."""
    items = [A.Resize(size, size)]
    if training:
        items += [
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.25),
            A.OneOf(
                [
                    A.ImageCompression(quality_range=(75, 98), p=1),
                    A.GaussianBlur((3, 3), (0.1, 0.8), p=1),
                    A.NoOp(p=1),
                ],
                p=0.35,
            ),
            A.ColorJitter(0.08, 0.08, 0.05, 0.01, p=0.25),
        ]
    return A.Compose(items)


class MaskDataset(Dataset):
    """Load RGB images and aligned binary manipulation masks.

    Attributes:
        frame: Reset data frame used as the row index.
        tf: Joint image and mask transformation pipeline."""

    def __init__(self, frame, size, training):
        """Initialize a segmentation dataset.

        Args:
            frame: Data frame containing image paths and polygon labels.
            size: Square model input resolution in pixels.
            training: Whether to enable stochastic augmentation."""
        self.frame = frame.reset_index(drop=True)
        self.tf = transforms(size, training)

    def __len__(self):
        """Return the number of available images.

        Returns:
            Number of rows in the backing data frame."""
        return len(self.frame)

    def __getitem__(self, index):
        """Load and transform one training example.

        Args:
            index: Zero-based row index.

        Returns:
            Tuple containing image tensor, mask tensor, image label, and row index."""
        row = self.frame.iloc[index]
        with Image.open(row.path) as source:
            image = np.asarray(source.convert("RGB"))
        mask = polygon_to_mask(row.polygon, *image.shape[:2])
        sample = self.tf(image=image, mask=mask)
        rgb = sample["image"].astype(np.float32) / 255.0
        rgb = (rgb - MEAN) / STD
        return (
            torch.from_numpy(rgb.transpose(2, 0, 1)).float(),
            torch.from_numpy(sample["mask"][None].astype(np.float32)),
            torch.tensor(float(row.has_mask)),
            index,
        )


def make_loader(frame, size, training, batch):
    """Create a deterministic data loader.

    Args:
        frame: Split data frame consumed by the loader.
        size: Square model input resolution in pixels.
        training: Whether to use augmentation and balanced sampling.
        batch: Physical number of images per batch.

    Returns:
        Configured PyTorch data loader."""
    dataset = MaskDataset(frame, size, training)
    generator = torch.Generator().manual_seed(SEED)
    args = dict(
        dataset=dataset,
        batch_size=batch,
        num_workers=2,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=True,
    )
    if training:
        pos = frame.has_mask.to_numpy(bool)
        weights = np.where(pos, 0.5 / max(1, pos.sum()), 0.5 / max(1, (~pos).sum()))
        args["sampler"] = WeightedRandomSampler(
            weights, len(weights), replacement=True, generator=generator
        )
    else:
        args["shuffle"] = False
    return DataLoader(**args)


class SRMBranch(nn.Module):
    """Extract a learned pyramid from fixed forensic residuals.

    Attributes:
        corrected: Whether to use true luminance and reflection padding.
        kernels: Three fixed high-pass SRM kernels.
        blocks: Convolutional residual feature stages."""

    def __init__(self, corrected=False):
        """Initialize the forensic residual branch.

        Args:
            corrected: Whether to apply the V3 photometric corrections."""
        super().__init__()
        self.corrected = corrected
        kernels = torch.tensor(
            [
                [
                    [0, 0, 0, 0, 0],
                    [0, -1, 2, -1, 0],
                    [0, 2, -4, 2, 0],
                    [0, -1, 2, -1, 0],
                    [0, 0, 0, 0, 0],
                ],
                [
                    [-1, 2, -2, 2, -1],
                    [2, -6, 8, -6, 2],
                    [-2, 8, -12, 8, -2],
                    [2, -6, 8, -6, 2],
                    [-1, 2, -2, 2, -1],
                ],
                [
                    [0, 0, 0, 0, 0],
                    [0, 0, -1, 0, 0],
                    [0, -1, 4, -1, 0],
                    [0, 0, -1, 0, 0],
                    [0, 0, 0, 0, 0],
                ],
            ],
            dtype=torch.float32,
        )
        kernels[0] /= 4
        kernels[1] /= 12
        kernels[2] /= 4
        self.register_buffer("kernels", kernels[:, None])
        channels = [32, 64, 128, 256]
        blocks = []
        incoming = 3
        for i, out in enumerate(channels):
            blocks.append(
                nn.Sequential(
                    nn.Conv2d(incoming, out, 3, 2, 1),
                    nn.BatchNorm2d(out),
                    nn.GELU(),
                    nn.Conv2d(out, out, 3, 2 if i == 0 else 1, 1),
                    nn.BatchNorm2d(out),
                    nn.GELU(),
                )
            )
            incoming = out
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x):
        """Extract multi-scale residual features.

        Args:
            x: Normalized RGB tensor shaped as ``(batch, 3, height, width)``.

        Returns:
            Four residual feature tensors ordered from fine to coarse."""
        if self.corrected:
            raw = (
                x * torch.as_tensor(STD, device=x.device)[None, :, None, None]
                + torch.as_tensor(MEAN, device=x.device)[None, :, None, None]
            )
            gray = 0.299 * raw[:, 0:1] + 0.587 * raw[:, 1:2] + 0.114 * raw[:, 2:3]
            residual = F.conv2d(F.pad(gray, (2, 2, 2, 2), mode="reflect"), self.kernels)
        else:
            residual = F.conv2d(x.mean(1, keepdim=True), self.kernels, padding=2)
        outputs = []
        for block in self.blocks:
            residual = block(residual)
            outputs.append(residual)
        return outputs


class ForensicFPN(nn.Module):
    """Fuse RGB semantics and SRM traces for localization.

    Attributes:
        enhanced: Whether V3 auxiliary and refinement heads are enabled.
        encoder: Pre-trained ConvNeXt-V2 feature extractor.
        srm: Fixed-filter forensic feature branch."""

    def __init__(self, enhanced=False):
        """Initialize the baseline or enhanced network.

        Args:
            enhanced: Whether to construct the V3-specific heads and classifier."""
        super().__init__()
        self.enhanced = enhanced
        self.encoder = timm.create_model(
            "convnextv2_tiny.fcmae_ft_in22k_in1k_384",
            pretrained=True,
            features_only=True,
            out_indices=(0, 1, 2, 3),
        )
        if hasattr(self.encoder, "set_grad_checkpointing"):
            self.encoder.set_grad_checkpointing(True)
        self.srm = SRMBranch(corrected=enhanced)
        channels = self.encoder.feature_info.channels()
        self.lateral = nn.ModuleList(
            [nn.Conv2d(a + b, 128, 1) for a, b in zip(channels, [32, 64, 128, 256])]
        )
        self.smooth = nn.ModuleList(
            [
                nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
                for _ in range(4)
            ]
        )
        self.mask_head = nn.Sequential(
            nn.Conv2d(512, 128, 3, padding=1),
            nn.GELU(),
            nn.Dropout2d(0.1),
            nn.Conv2d(128, 1, 1),
        )
        self.boundary_head = nn.Sequential(
            nn.Conv2d(512, 64, 3, padding=1), nn.GELU(), nn.Conv2d(64, 1, 1)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.LayerNorm(128),
            nn.Dropout(0.25),
            nn.Linear(128, 1),
        )
        self.rgb_classifier = (
            nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.LayerNorm(channels[-1]),
                nn.Dropout(0.25),
                nn.Linear(channels[-1], 1),
            )
            if enhanced
            else None
        )
        self.aux_heads = (
            nn.ModuleList([nn.Conv2d(128, 1, 1) for _ in range(3)])
            if enhanced
            else nn.ModuleList()
        )
        self.refine = (
            nn.Sequential(
                nn.Conv2d(514, 128, 3, padding=1), nn.GELU(), nn.Conv2d(128, 1, 1)
            )
            if enhanced
            else None
        )

    def forward(self, image):
        """Predict mask, boundary, and image-level logits.

        Args:
            image: Normalized RGB batch shaped as ``(batch, 3, height, width)``.

        Returns:
            Dictionary containing final mask, boundary, class, and auxiliary logits."""
        semantic, residual = self.encoder(image), self.srm(image)
        pyramid = [
            layer(torch.cat([a, b], 1))
            for layer, a, b in zip(self.lateral, semantic, residual)
        ]
        for i in range(2, -1, -1):
            pyramid[i] = pyramid[i] + F.interpolate(
                pyramid[i + 1],
                pyramid[i].shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        pyramid = [layer(x) for layer, x in zip(self.smooth, pyramid)]
        target = pyramid[0].shape[-2:]
        fused = torch.cat(
            [
                (
                    x
                    if x.shape[-2:] == target
                    else F.interpolate(x, target, mode="bilinear", align_corners=False)
                )
                for x in pyramid
            ],
            1,
        )
        base = self.mask_head(fused)
        boundary = self.boundary_head(fused)
        mask = (
            self.refine(torch.cat([fused, base, boundary], 1))
            if self.enhanced
            else base
        )
        size = image.shape[-2:]
        forensic_class = self.classifier(pyramid[-1]).squeeze(1)
        class_logits = (
            0.5 * (forensic_class + self.rgb_classifier(semantic[-1]).squeeze(1))
            if self.enhanced
            else forensic_class
        )
        return {
            "mask": F.interpolate(mask, size, mode="bilinear", align_corners=False),
            "boundary": F.interpolate(
                boundary, size, mode="bilinear", align_corners=False
            ),
            "class": class_logits,
            "aux": [
                F.interpolate(h(pyramid[i]), size, mode="bilinear", align_corners=False)
                for i, h in enumerate(self.aux_heads)
            ],
        }


## 4. Losses, EMA, snapshots et TTA


In [6]:
def boundary_target(mask):
    """Build a narrow morphological boundary target.

    Args:
        mask: Binary mask tensor shaped as ``(batch, 1, height, width)``.

    Returns:
        Binary tensor marking pixels around mask boundaries."""
    return (F.max_pool2d(mask, 3, 1, 1) + F.max_pool2d(-mask, 3, 1, 1)).clamp(0, 1)


def focal_bce(logits, target, gamma=2):
    """Compute focal binary cross-entropy.

    Args:
        logits: Unnormalized binary predictions.
        target: Binary target tensor matching ``logits``.
        gamma: Focusing exponent applied to easy examples.

    Returns:
        Scalar focal loss."""
    base = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    p = torch.sigmoid(logits)
    pt = p * target + (1 - p) * (1 - target)
    return ((1 - pt).pow(gamma) * base).mean()


def soft_iou(logits, target, positive):
    """Compute differentiable IoU loss on positive images.

    Args:
        logits: Pixel-level mask logits.
        target: Binary mask targets.
        positive: Boolean vector selecting positive images.

    Returns:
        Scalar soft-IoU loss, or differentiable zero without positives."""
    if not positive.any():
        return logits.sum() * 0
    p = torch.sigmoid(logits[positive]).flatten(1)
    t = target[positive].flatten(1)
    inter = (p * t).sum(1)
    union = p.sum(1) + t.sum(1) - inter
    return (1 - (inter + 1) / (union + 1)).mean()


def lovasz_hinge_flat(logits, labels):
    """Compute binary Lovasz hinge on flattened tensors.

    Args:
        logits: Flattened mask logits.
        labels: Flattened binary labels.

    Returns:
        Scalar Lovasz surrogate for intersection over union."""
    if labels.numel() == 0:
        return logits.sum() * 0
    signs = 2 * labels.float() - 1
    errors = 1 - logits * signs
    errors, perm = torch.sort(errors, descending=True)
    labels = labels[perm]
    gts = labels.sum()
    intersection = gts - labels.float().cumsum(0)
    union = gts + (1 - labels).float().cumsum(0)
    grad = 1 - intersection / union.clamp_min(1)
    if len(grad) > 1:
        # fix : grad[1:] et grad[:-1] se chevauchent (vues du même tenseur) ;
        # la soustraction in-place n'a pas d'ordre garanti sur des tranches
        # qui se recouvrent (RuntimeError explicite de PyTorch). On clone la
        # tranche lue pour figer ses valeurs AVANT l'écriture, ce qui restaure
        # exactement la sémantique voulue (diff discret) sans rien changer au
        # calcul lui-même.
        grad[1:] -= grad[:-1].clone()
    return torch.dot(F.relu(errors), grad)


def balanced_boundary(logits, target):
    """Compute class-balanced BCE and Dice boundary loss.

    Args:
        logits: Boundary logits.
        target: Binary boundary targets.

    Returns:
        Scalar boundary loss."""
    positives = target.sum()
    negatives = target.numel() - positives
    weight = (negatives / positives.clamp_min(1)).clamp(1, 25)
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=weight)
    p = torch.sigmoid(logits)
    dice = 1 - (2 * (p * target).sum() + 1) / (p.sum() + target.sum() + 1)
    return bce + dice


def combined_loss(outputs, mask, label, enhanced):
    """Combine localization and image-level objectives.

    Args:
        outputs: Model output dictionary.
        mask: Binary ground-truth masks.
        label: Binary image-level manipulation labels.
        enhanced: Whether to include V3-specific auxiliary objectives.

    Returns:
        Scalar loss used for backpropagation."""
    positive = label > 0.5
    total = (
        0.65 * focal_bce(outputs["mask"], mask)
        + 0.85 * soft_iou(outputs["mask"], mask, positive)
        + 0.35 * F.binary_cross_entropy_with_logits(outputs["class"], label)
    )
    if enhanced:
        small_logits = F.interpolate(
            outputs["mask"], scale_factor=0.25, mode="bilinear", align_corners=False
        )
        small_truth = F.interpolate(mask, small_logits.shape[-2:], mode="nearest")
        lovasz = torch.stack(
            [
                lovasz_hinge_flat(x.flatten(), y.flatten())
                for x, y in zip(small_logits[:, 0], small_truth[:, 0])
            ]
        ).mean()
        aux = sum(
            focal_bce(x, mask) + 0.25 * soft_iou(x, mask, positive)
            for x in outputs["aux"]
        ) / max(1, len(outputs["aux"]))
        total += (
            0.20 * lovasz
            + 0.15 * balanced_boundary(outputs["boundary"], boundary_target(mask))
            + 0.12 * aux
        )
    else:
        total += 0.15 * F.binary_cross_entropy_with_logits(
            outputs["boundary"], boundary_target(mask)
        )
    return total


@torch.no_grad()
def update_ema(ema, model, decay=0.995):
    """Update exponential moving-average model parameters.

    Args:
        ema: Model receiving smoothed parameters in place.
        model: Current optimized model.
        decay: Weight retained from the previous moving average."""
    source = model.state_dict()
    for name, value in ema.state_dict().items():
        if value.dtype.is_floating_point:
            value.mul_(decay).add_(source[name].detach(), alpha=1 - decay)
        else:
            value.copy_(source[name])


def cpu_state(model):
    """Copy a model state dictionary to CPU memory.

    Args:
        model: Model whose tensors are copied.

    Returns:
        Detached CPU state dictionary."""
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def average_states(states):
    """Average floating tensors across model snapshots.

    Args:
        states: Non-empty sequence of compatible state dictionaries.

    Returns:
        State dictionary with averaged floating tensors and latest buffers."""
    result = {}
    for key in states[0]:
        values = [s[key] for s in states]
        result[key] = (
            torch.stack(values).mean(0)
            if values[0].dtype.is_floating_point
            else values[-1]
        )
    return result


def transform_image(x, index):
    """Apply one dihedral test-time transformation.

    Args:
        x: Batched image tensor.
        index: Transformation index from zero through seven.

    Returns:
        Transformed image tensor."""
    rotation = index // 2
    flip = index % 2
    y = torch.rot90(x, rotation, (-2, -1))
    return torch.flip(y, (-1,)) if flip else y


def invert_map(x, index):
    """Undo one dihedral transformation on a prediction map.

    Args:
        x: Batched prediction tensor.
        index: Transformation index originally applied to the input.

    Returns:
        Prediction tensor restored to the input orientation."""
    rotation = index // 2
    flip = index % 2
    y = torch.flip(x, (-1,)) if flip else x
    return torch.rot90(y, -rotation, (-2, -1))


@torch.no_grad()
def predict(model, loader, views=2):
    """Collect TTA mask and class probabilities in row order.

    Args:
        model: Trained localization model.
        loader: Deterministic evaluation data loader.
        views: Number of dihedral views, either two or eight.

    Returns:
        Tuple of mask probability maps and image-level probabilities."""
    model.eval()
    maps = [None] * len(loader.dataset)
    classes = np.zeros(len(loader.dataset), np.float32)
    for image, _, _, indices in loader:
        image = image.to(DEVICE, non_blocking=True)
        map_sum = 0
        class_sum = 0
        for view in range(views):
            augmented = transform_image(image, view if views == 8 else view)
            with torch.autocast("cuda", dtype=torch.float16):
                output = model(augmented)
            map_sum += invert_map(
                torch.sigmoid(output["mask"]), view if views == 8 else view
            )
            class_sum += torch.sigmoid(output["class"])
        probability = (map_sum / views)[:, 0].float().cpu().numpy()
        class_probability = (class_sum / views).float().cpu().numpy()
        for offset, index in enumerate(indices.numpy()):
            maps[index] = cv2.resize(
                probability[offset],
                (EVAL_SIZE, EVAL_SIZE),
                interpolation=cv2.INTER_LINEAR,
            ).astype(np.float16)
            classes[index] = class_probability[offset]
    return maps, classes


## 5. Entraînement comparatif V2/V3


In [7]:
def safe_batch(enhanced, size):
    """Select the largest batch that completes backpropagation.

    Args:
        enhanced: Whether to probe the enhanced V3 architecture.
        size: Square input resolution in pixels.

    Returns:
        Largest successful physical batch from four, two, and one.

    Raises:
        RuntimeError: If a single image does not fit in GPU memory."""
    for batch in (4, 2, 1):
        resources = {}
        try:
            resources["model"] = ForensicFPN(enhanced).to(DEVICE)
            sample = torch.zeros(batch, 3, size, size, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = resources["model"](sample)
                loss = combined_loss(
                    outputs,
                    torch.zeros(batch, 1, size, size, device=DEVICE),
                    torch.zeros(batch, device=DEVICE),
                    enhanced,
                )
                loss.backward()
            del sample, outputs, loss
            resources.clear()
            torch.cuda.empty_cache()
            return batch
        except torch.cuda.OutOfMemoryError:
            resources.clear()
            gc.collect()
            torch.cuda.empty_cache()
    raise RuntimeError("OOM même au batch 1")


def train_development(name, enhanced, size):
    """Train one development candidate with EMA snapshots.

    Args:
        name: Human-readable candidate name stored in the audit.
        enhanced: Whether to train the V3 architecture.
        size: Square training resolution in pixels.

    Returns:
        Candidate metadata, histories, and final CPU snapshot states."""
    batch = safe_batch(enhanced, size)
    accumulation = max(1, 16 // batch)
    loader = make_loader(fit_df, size, True, batch)
    model = ForensicFPN(enhanced).to(DEVICE)
    ema = copy.deepcopy(model).eval()
    for parameter in ema.parameters():
        parameter.requires_grad = False
    encoder = list(model.encoder.parameters())
    ids = {id(x) for x in encoder}
    new = [x for x in model.parameters() if id(x) not in ids]
    optimizer = torch.optim.AdamW(
        [{"params": encoder, "lr": 3e-5}, {"params": new, "lr": 1.2e-4}],
        weight_decay=2e-4,
    )
    steps = math.ceil(len(loader) / accumulation)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, max(1, steps * DEV_EPOCHS)
    )
    scaler = torch.amp.GradScaler("cuda")
    snapshots = []
    history = []
    for epoch in range(1, DEV_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for step, (image, mask, label, _) in enumerate(loader, 1):
            image, mask, label = image.to(DEVICE), mask.to(DEVICE), label.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = combined_loss(model(image), mask, label, enhanced) / accumulation
            scaler.scale(loss).backward()
            if step % accumulation == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_ema(ema, model)
            losses.append(float(loss.detach().cpu()) * accumulation)
        history.append({"epoch": epoch, "loss": float(np.mean(losses))})
        print(name, history[-1])
        if epoch >= DEV_EPOCHS - 2:
            snapshots.append(cpu_state(ema))
    last = snapshots[-1]
    averaged = average_states(snapshots)
    del model, ema, loader
    gc.collect()
    torch.cuda.empty_cache()
    return {
        "name": name,
        "enhanced": enhanced,
        "size": size,
        "batch": batch,
        "last": last,
        "average": averaged,
        "history": history,
    }


BASELINE = train_development("v2_baseline", False, BASE_SIZE)
PHASE_TIMES["baseline_development_hours"] = elapsed_hours()
if elapsed_hours() >= MAX_COMPARE_HOURS:
    raise RuntimeError("Budget comparaison épuisé avant V3")
V3 = train_development("v3_enhanced", True, V3_SIZE)
PHASE_TIMES["v3_development_hours"] = (
    elapsed_hours() - PHASE_TIMES["baseline_development_hours"]
)


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

v2_baseline {'epoch': 1, 'loss': 0.4385102325740847}
v2_baseline {'epoch': 2, 'loss': 0.275819028865761}
v2_baseline {'epoch': 3, 'loss': 0.23222575177719984}
v2_baseline {'epoch': 4, 'loss': 0.20038336903223214}
v2_baseline {'epoch': 5, 'loss': 0.1839840190372241}
v2_baseline {'epoch': 6, 'loss': 0.1810171478455199}
v3_enhanced {'epoch': 1, 'loss': 0.7875500680656413}
v3_enhanced {'epoch': 2, 'loss': 0.5302309404214507}
v3_enhanced {'epoch': 3, 'loss': 0.45200971893551295}
v3_enhanced {'epoch': 4, 'loss': 0.4010098547069922}
v3_enhanced {'epoch': 5, 'loss': 0.3665698587744989}
v3_enhanced {'epoch': 6, 'loss': 0.37110003615464726}


## 6. Retrieval et calibration native


In [8]:
def build_hash_index(frame, hashes):
    """Index empty reference images by perceptual hash chunks.

    Args:
        frame: Gallery data frame containing the ``has_mask`` column.
        hashes: Perceptual hashes aligned with gallery rows.

    Returns:
        Bucket dictionary mapping hash chunks to gallery candidates."""
    buckets = {}
    for index, (positive, value) in enumerate(zip(frame.has_mask, hashes)):
        if positive:
            continue
        integer = int(value, 16)
        for pos, shift in enumerate((56, 48, 40, 32, 24, 16, 8, 0)):
            buckets.setdefault((pos, (integer >> shift) & 255), []).append(
                (index, integer)
            )
    return buckets


def retrieval_probability(path, target_hash, gallery, buckets):
    """Estimate changes against an aligned empty reference.

    Args:
        path: Path of the target image.
        target_hash: Perceptual hash of the target image.
        gallery: Data frame containing empty reference candidates.
        buckets: Chunked perceptual-hash index for the gallery.

    Returns:
        Tuple containing a change probability map and retrieval confidence."""
    integer = int(target_hash, 16)
    candidates = {}
    for pos, shift in enumerate((56, 48, 40, 32, 24, 16, 8, 0)):
        for index, other in buckets.get((pos, (integer >> shift) & 255), []):
            candidates[index] = other
    ranked = sorted(
        ((integer ^ other).bit_count(), index) for index, other in candidates.items()
    )[:4]
    if not ranked or ranked[0][0] > 10:
        return np.zeros((EVAL_SIZE, EVAL_SIZE), np.float16), 0.0
    target = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    target = cv2.resize(target, (EVAL_SIZE, EVAL_SIZE))
    gray = cv2.cvtColor(target, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255
    small = cv2.resize(gray, (256, 256))
    best = np.zeros((EVAL_SIZE, EVAL_SIZE), np.float32)
    best_conf = 0.0
    for distance, index in ranked:
        ref = cv2.cvtColor(cv2.imread(str(gallery.path.iloc[index])), cv2.COLOR_BGR2RGB)
        ref = cv2.resize(ref, (EVAL_SIZE, EVAL_SIZE))
        ref_gray = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255
        warp = np.eye(2, 3, dtype=np.float32)
        try:
            correlation, warp = cv2.findTransformECC(
                small,
                cv2.resize(ref_gray, (256, 256)),
                warp,
                cv2.MOTION_AFFINE,
                (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 25, 1e-4),
                None,
                3,
            )
        except cv2.error:
            continue
        warp[:, 2] *= EVAL_SIZE / 256
        aligned = cv2.warpAffine(
            ref,
            warp,
            (EVAL_SIZE, EVAL_SIZE),
            flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
            borderMode=cv2.BORDER_REFLECT,
        )
        color = (
            np.mean(
                np.abs(
                    cv2.cvtColor(target, cv2.COLOR_RGB2LAB).astype(np.float32)
                    - cv2.cvtColor(aligned, cv2.COLOR_RGB2LAB).astype(np.float32)
                ),
                2,
            )
            / 45
        )
        edge = (
            np.abs(
                cv2.Laplacian(gray, cv2.CV_32F)
                - cv2.Laplacian(
                    cv2.cvtColor(aligned, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255,
                    cv2.CV_32F,
                )
            )
            / 0.35
        )
        probability = cv2.GaussianBlur(
            np.clip(0.75 * color + 0.25 * edge, 0, 1), (0, 0), 2
        )
        confidence = max(0, float(correlation)) * max(0, 1 - distance / 12)
        if confidence > best_conf:
            best, best_conf = probability, confidence
    return (
        (best.astype(np.float16), best_conf)
        if best_conf >= 0.35
        else (np.zeros_like(best, np.float16), 0.0)
    )


lookup = {
    row.image_id: value for (_, row), value in zip(train_df.iterrows(), train_hashes)
}
gallery = pd.concat([fit_df, selection_df], ignore_index=True)
gallery_hashes = [lookup[x] for x in gallery.image_id]
calib_hashes = [lookup[x] for x in calib_df.image_id]
buckets = build_hash_index(gallery, gallery_hashes)
calib_retrieval = []
calib_conf = []
for (_, row), value in zip(calib_df.iterrows(), calib_hashes):
    probability, confidence = retrieval_probability(row.path, value, gallery, buckets)
    calib_retrieval.append(probability)
    calib_conf.append(confidence)
calib_conf = np.asarray(calib_conf, np.float32)


def remove_small(mask, area):
    """Remove connected components below an area threshold.

    Args:
        mask: Binary prediction mask.
        area: Minimum retained component area in pixels.

    Returns:
        Cleaned unsigned binary mask."""
    if area <= 0 or not mask.any():
        return mask.astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    result = np.zeros_like(mask, np.uint8)
    for i in range(1, count):
        if stats[i, cv2.CC_STAT_AREA] >= area:
            result[labels == i] = 1
    return result


def top_evidence(probability):
    """Summarize the strongest 0.1 percent of mask probabilities.

    Args:
        probability: Two-dimensional mask probability map.

    Returns:
        Mean probability among the selected highest pixels."""
    flat = np.asarray(probability, np.float32).ravel()
    count = max(1, int(len(flat) * 0.001))
    return float(np.partition(flat, -count)[-count:].mean())


TRUTH_CACHE = {}


def native_iou(frame, index, mask, epsilon):
    """Score one prediction after native-resolution round-trip.

    Args:
        frame: Data frame containing polygons and original dimensions.
        index: Row index of the prediction.
        mask: Predicted binary mask at the common evaluation size.
        epsilon: Relative contour simplification factor.

    Returns:
        Official IoU at the image's original resolution."""
    row = frame.iloc[index]
    polygon = mask_to_polygon(mask, epsilon, 3)
    if row.image_id not in TRUTH_CACHE:
        TRUTH_CACHE[row.image_id] = polygon_to_mask(row.polygon, row.height, row.width)
    return official_iou(
        TRUTH_CACHE[row.image_id], polygon_to_mask(polygon, row.height, row.width)
    )


def evidence_values(maps, classes, alpha):
    """Blend image-level and localized manipulation evidence.

    Args:
        maps: Mask probability maps.
        classes: Image-level manipulation probabilities.
        alpha: Weight assigned to image-level probabilities.

    Returns:
        One blended evidence value per image."""
    return np.asarray(
        [alpha * c + (1 - alpha) * top_evidence(m) for m, c in zip(maps, classes)]
    )


def evaluate_config(
    frame, maps, classes, retrieval, confidence, config, return_masks=False
):
    """Evaluate one calibrated inference configuration.

    Args:
        frame: Labelled split data frame.
        maps: Neural mask probability maps.
        classes: Image-level probabilities.
        retrieval: Retrieval change maps.
        confidence: Retrieval confidence values.
        config: Gating, threshold, morphology, and codec configuration.
        return_masks: Whether to include detailed masks and scores.

    Returns:
        Mean IoU alone, or detailed score data when requested."""
    evidence = evidence_values(maps, classes, config["alpha"])
    threshold = (
        np.partition(evidence, -round(len(evidence) / 3))[-round(len(evidence) / 3)]
        if config["gate_mode"] == "rank"
        else config["class_threshold"]
    )
    scores = []
    outputs = []
    for i, (neural, ret, conf) in enumerate(zip(maps, retrieval, confidence)):
        if evidence[i] < threshold:
            mask = np.zeros((EVAL_SIZE, EVAL_SIZE), np.uint8)
        else:
            weight = config["retrieval_weight"] if conf >= 0.35 else 0.0
            fused = (1 - weight) * np.asarray(neural, np.float32) + weight * np.asarray(
                ret, np.float32
            )
            mask = remove_small(
                fused >= config["pixel_threshold"], config["minimum_area"]
            )
            if config["closing"]:
                mask = cv2.morphologyEx(
                    mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8)
                )
        scores.append(native_iou(frame, i, mask, config.get("epsilon", 0.0)))
        outputs.append(mask)
    return (
        (float(np.mean(scores)), np.asarray(scores), outputs, evidence, threshold)
        if return_masks
        else float(np.mean(scores))
    )


def calibrate(frame, maps, classes, retrieval, confidence):
    """Select inference parameters on one calibration split.

    Args:
        frame: Calibration data frame.
        maps: Neural mask probability maps.
        classes: Image-level probabilities.
        retrieval: Retrieval change maps.
        confidence: Retrieval confidence values.

    Returns:
        Best score, selected configuration, and epsilon score mapping."""
    candidates = []
    for alpha in (1.0, 0.85, 0.70):
        for gate_mode in ("threshold", "rank"):
            for class_threshold in (
                (0.4, 0.5, 0.6) if gate_mode == "threshold" else (0.5,)
            ):
                for pixel_threshold in (0.45, 0.50, 0.55, 0.60):
                    for minimum_area in (0, 128):
                        for retrieval_weight in (0.0, 0.25):
                            for closing in (False, True):
                                config = dict(
                                    alpha=alpha,
                                    gate_mode=gate_mode,
                                    class_threshold=class_threshold,
                                    pixel_threshold=pixel_threshold,
                                    minimum_area=minimum_area,
                                    retrieval_weight=retrieval_weight,
                                    closing=closing,
                                    epsilon=0.0,
                                )
                                candidates.append(
                                    (
                                        evaluate_config(
                                            frame,
                                            maps,
                                            classes,
                                            retrieval,
                                            confidence,
                                            config,
                                        ),
                                        config,
                                    )
                                )
    score, config = max(candidates, key=lambda x: x[0])
    if config["retrieval_weight"] > 0:
        without = {**config, "retrieval_weight": 0.0}
        score_without = evaluate_config(
            frame, maps, classes, retrieval, confidence, without
        )
        if score - score_without < MIN_RETRIEVAL_GAIN:
            score, config = score_without, without
    epsilon_scores = {
        e: evaluate_config(
            frame, maps, classes, retrieval, confidence, {**config, "epsilon": e}
        )
        for e in (0.0, 0.0015, 0.0025)
    }
    config["epsilon"] = max(epsilon_scores, key=epsilon_scores.get)
    return epsilon_scores[config["epsilon"]], config, epsilon_scores


## 7. Sélection automatique et validation officielle


In [9]:
def load_candidate(candidate, state):
    """Instantiate a candidate and restore one CPU state.

    Args:
        candidate: Candidate metadata containing architecture settings.
        state: Compatible model state dictionary.

    Returns:
        Restored model placed on the configured GPU."""
    model = ForensicFPN(candidate["enhanced"]).to(DEVICE)
    model.load_state_dict(state)
    return model


def assess_candidate(candidate):
    """Select a snapshot and calibrate one architecture.

    Args:
        candidate: Development candidate returned by training.

    Returns:
        Selection and calibration metrics with probability maps."""
    selection_loader = make_loader(
        selection_df, candidate["size"], False, max(1, candidate["batch"] * 2)
    )
    variants = []
    reference_config = dict(
        alpha=1.0,
        gate_mode="threshold",
        class_threshold=0.5,
        pixel_threshold=0.45,
        minimum_area=0,
        retrieval_weight=0.0,
        closing=False,
        epsilon=0.0,
    )
    zero_retrieval = [np.zeros((EVAL_SIZE, EVAL_SIZE), np.float16)] * len(selection_df)
    zero_conf = np.zeros(len(selection_df), np.float32)
    for state_name in ("last", "average"):
        model = load_candidate(candidate, candidate[state_name])
        maps, classes = predict(model, selection_loader, 2)
        detailed = evaluate_config(
            selection_df,
            maps,
            classes,
            zero_retrieval,
            zero_conf,
            reference_config,
            True,
        )
        labels = selection_df.has_mask.to_numpy(np.uint8)
        predicted = np.asarray([m.any() for m in detailed[2]], np.uint8)
        matrix = confusion_matrix(labels, predicted, labels=[0, 1])
        variants.append(
            dict(
                state=state_name,
                selection_score=detailed[0],
                selection_fpr=float(matrix[0, 1] / max(1, matrix[0].sum())),
                selection_matrix=matrix.tolist(),
            )
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()
    result = max(variants, key=lambda x: x["selection_score"])
    model = load_candidate(candidate, candidate[result["state"]])
    calib_loader = make_loader(
        calib_df, candidate["size"], False, max(1, candidate["batch"] * 2)
    )
    maps, classes = predict(model, calib_loader, 2)
    del model, calib_loader, selection_loader
    gc.collect()
    torch.cuda.empty_cache()
    score, config, eps = calibrate(calib_df, maps, classes, calib_retrieval, calib_conf)
    detailed = evaluate_config(
        calib_df, maps, classes, calib_retrieval, calib_conf, config, True
    )
    labels = calib_df.has_mask.to_numpy(np.uint8)
    predicted = np.asarray([m.any() for m in detailed[2]], np.uint8)
    matrix = confusion_matrix(labels, predicted, labels=[0, 1])
    result.update(
        score=score,
        config=config,
        epsilon_scores=eps,
        maps=maps,
        classes=classes,
        calibration_fpr=float(matrix[0, 1] / max(1, matrix[0].sum())),
        calibration_matrix=matrix.tolist(),
    )
    return result


BASE_RESULT = assess_candidate(BASELINE)
V3_RESULT = assess_candidate(V3)
v3_allowed = (
    V3_RESULT["selection_score"] >= BASE_RESULT["selection_score"] + MIN_V3_GAIN
    and V3_RESULT["selection_fpr"] <= BASE_RESULT["selection_fpr"] + 1e-12
)
SELECTED, SELECTED_RESULT = (V3, V3_RESULT) if v3_allowed else (BASELINE, BASE_RESULT)
print(
    "selection baseline/v3",
    BASE_RESULT["selection_score"],
    V3_RESULT["selection_score"],
    "calibration",
    BASE_RESULT["score"],
    V3_RESULT["score"],
    "selected",
    SELECTED["name"],
)
PHASE_TIMES["comparison_and_calibration_cumulative_hours"] = elapsed_hours()

# Le TTA 8 vues doit prouver son utilité sur calibration.
selected_model = load_candidate(SELECTED, SELECTED[SELECTED_RESULT["state"]])
selected_loader = make_loader(
    calib_df, SELECTED["size"], False, max(1, SELECTED["batch"] * 2)
)
FINAL_VIEWS = 2
DEV_MAPS, DEV_CLASSES = SELECTED_RESULT["maps"], SELECTED_RESULT["classes"]
FINAL_CONFIG = SELECTED_RESULT["config"]
DEV_SCORE = SELECTED_RESULT["score"]
if elapsed_hours() < MAX_REFIT_START_HOURS - 0.35:
    d4_maps, d4_classes = predict(selected_model, selected_loader, 8)
    d4_reference = evaluate_config(
        calib_df, d4_maps, d4_classes, calib_retrieval, calib_conf, FINAL_CONFIG
    )
    if d4_reference >= DEV_SCORE + MIN_TTA_GAIN / 2:
        d4_score, d4_config, d4_eps = calibrate(
            calib_df, d4_maps, d4_classes, calib_retrieval, calib_conf
        )
        if d4_score >= DEV_SCORE + MIN_TTA_GAIN:
            FINAL_VIEWS = 8
            DEV_MAPS, DEV_CLASSES = d4_maps, d4_classes
            FINAL_CONFIG = d4_config
            DEV_SCORE = d4_score
else:
    WARNINGS.append("TTA 8 vues non testé afin de réserver le temps du refit")
del selected_model
gc.collect()
torch.cuda.empty_cache()

# Validation officielle ouverte une seule fois après tous les choix.
train_buckets = build_hash_index(train_df, train_hashes)
val_hashes = compute_hashes(val_df, f"{RUN_TAG}_val_phash.npy")
val_retrieval = []
val_conf = []
for (_, row), value in zip(val_df.iterrows(), val_hashes):
    p, c = retrieval_probability(row.path, value, train_df, train_buckets)
    val_retrieval.append(p)
    val_conf.append(c)
audit_model = load_candidate(SELECTED, SELECTED[SELECTED_RESULT["state"]])
val_loader = make_loader(val_df, SELECTED["size"], False, max(1, SELECTED["batch"] * 2))
val_maps, val_classes = predict(audit_model, val_loader, FINAL_VIEWS)
val_detail = evaluate_config(
    val_df, val_maps, val_classes, val_retrieval, val_conf, FINAL_CONFIG, True
)
VAL_SCORE, VAL_IOUS, VAL_MASKS = val_detail[:3]
labels = val_df.has_mask.to_numpy(np.uint8)
predicted = np.asarray([m.any() for m in VAL_MASKS], np.uint8)
VAL_MATRIX = confusion_matrix(labels, predicted, labels=[0, 1]).tolist()
VAL_POSITIVE = float(VAL_IOUS[labels == 1].mean())
print("official val", VAL_SCORE, "positive", VAL_POSITIVE, "gate", VAL_MATRIX)
PHASE_TIMES["official_validation_cumulative_hours"] = elapsed_hours()
del audit_model, val_loader
gc.collect()
torch.cuda.empty_cache()


selection baseline/v3 0.9307188631971077 0.9392445263378243 calibration 0.9314703856557379 0.9385329343636843 selected v3_enhanced
official val 0.9385365519477794 positive 0.8217446251684912 gate [[650, 2], [2, 324]]


## 8. Refit final, soumission et audit


In [10]:
if elapsed_hours() >= MAX_REFIT_START_HOURS:
    raise RuntimeError("Refit trop tardif pour garantir un export propre")
all_df = pd.concat([train_df, val_df], ignore_index=True)
batch = SELECTED["batch"]
accumulation = max(1, 16 // batch)
loader = make_loader(all_df, SELECTED["size"], True, batch)
model = ForensicFPN(SELECTED["enhanced"]).to(DEVICE)
ema = copy.deepcopy(model).eval()
for p in ema.parameters():
    p.requires_grad = False
encoder = list(model.encoder.parameters())
ids = {id(x) for x in encoder}
new = [x for x in model.parameters() if id(x) not in ids]
optimizer = torch.optim.AdamW(
    [{"params": encoder, "lr": 3e-5}, {"params": new, "lr": 1.2e-4}], weight_decay=2e-4
)
steps = math.ceil(len(loader) / accumulation)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, max(1, steps * FINAL_EPOCHS)
)
scaler = torch.amp.GradScaler("cuda")
final_states = []
final_history = []
for epoch in range(1, FINAL_EPOCHS + 1):
    if epoch > 4 and elapsed_hours() >= MAX_START_EPOCH_HOURS:
        WARNINGS.append("Refit arrêté par garde temporelle")
        break
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses = []
    for step, (image, mask, label, _) in enumerate(loader, 1):
        image, mask, label = image.to(DEVICE), mask.to(DEVICE), label.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = (
                combined_loss(model(image), mask, label, SELECTED["enhanced"])
                / accumulation
            )
        scaler.scale(loss).backward()
        if step % accumulation == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            update_ema(ema, model)
        losses.append(float(loss.detach().cpu()) * accumulation)
    final_history.append({"epoch": epoch, "loss": float(np.mean(losses))})
    print("final", final_history[-1])
    if epoch >= 6:
        final_states.append(cpu_state(ema))
if len(final_history) < 4:
    raise RuntimeError("Refit final insuffisant")
PHASE_TIMES["final_training_cumulative_hours"] = elapsed_hours()
if SELECTED_RESULT["state"] == "average" and len(final_states) >= 2:
    ema.load_state_dict(average_states(final_states))
torch.save(
    {
        "model": cpu_state(ema),
        "architecture": SELECTED["name"],
        "epochs": len(final_history),
    },
    OUT / f"{RUN_TAG}_final.pt",
)
del model, loader
gc.collect()
torch.cuda.empty_cache()
if elapsed_hours() >= MAX_TOTAL_HOURS:
    raise RuntimeError("Budget épuisé avant inférence")
TEST_VIEWS = FINAL_VIEWS
if TEST_VIEWS == 8 and elapsed_hours() > 5.90:
    TEST_VIEWS = 2
    WARNINGS.append("TTA test réduit à deux vues par garde temporelle")

test_loader = make_loader(test_df, SELECTED["size"], False, max(1, batch * 2))
test_maps, test_classes = predict(ema, test_loader, TEST_VIEWS)
all_hashes = train_hashes + val_hashes
all_buckets = build_hash_index(all_df, all_hashes)
test_hashes = compute_hashes(test_df, f"{RUN_TAG}_test_phash.npy")
test_retrieval = []
test_conf = []
for (_, row), value in zip(test_df.iterrows(), test_hashes):
    p, c = retrieval_probability(row.path, value, all_df, all_buckets)
    test_retrieval.append(p)
    test_conf.append(c)

# Appliquer la configuration sans vérité terrain.
evidence = evidence_values(test_maps, test_classes, FINAL_CONFIG["alpha"])
gate_threshold = (
    np.partition(evidence, -round(len(evidence) / 3))[-round(len(evidence) / 3)]
    if FINAL_CONFIG["gate_mode"] == "rank"
    else FINAL_CONFIG["class_threshold"]
)
polygons = []
for i, (neural, ret, conf) in enumerate(zip(test_maps, test_retrieval, test_conf)):
    if evidence[i] < gate_threshold:
        mask = np.zeros((EVAL_SIZE, EVAL_SIZE), np.uint8)
    else:
        weight = FINAL_CONFIG["retrieval_weight"] if conf >= 0.35 else 0.0
        fused = (1 - weight) * np.asarray(neural, np.float32) + weight * np.asarray(
            ret, np.float32
        )
        mask = remove_small(
            fused >= FINAL_CONFIG["pixel_threshold"], FINAL_CONFIG["minimum_area"]
        )
        if FINAL_CONFIG["closing"]:
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    polygons.append(mask_to_polygon(mask, FINAL_CONFIG["epsilon"], 3))
submission = pd.DataFrame({"image_id": test_df.image_id, "polygon": polygons})
assert (
    len(submission) == len(sample_submission) == 993
    and not submission.image_id.duplicated().any()
    and set(submission.image_id) == set(sample_submission.image_id)
)
submission.to_csv(OUT / "submission.csv", index=False)
PHASE_TIMES["test_inference_and_retrieval_cumulative_hours"] = elapsed_hours()

comparison = pd.DataFrame(
    [
        {
            "candidate": "v2",
            "selection_score": BASE_RESULT["selection_score"],
            "selection_fpr": BASE_RESULT["selection_fpr"],
            "calibration_score": BASE_RESULT["score"],
            "calibration_fpr": BASE_RESULT["calibration_fpr"],
            "state": BASE_RESULT["state"],
        },
        {
            "candidate": "v3",
            "selection_score": V3_RESULT["selection_score"],
            "selection_fpr": V3_RESULT["selection_fpr"],
            "calibration_score": V3_RESULT["score"],
            "calibration_fpr": V3_RESULT["calibration_fpr"],
            "state": V3_RESULT["state"],
        },
    ]
)
comparison.to_csv(OUT / "candidate_comparison.csv", index=False)
maps_dir = OUT / f"{RUN_TAG}_maps"
maps_dir.mkdir(exist_ok=True)
for name, values in (("calib", DEV_MAPS), ("val", val_maps), ("test", test_maps)):
    np.save(
        maps_dir / f"{name}_mask_uint8.npy",
        np.rint(np.asarray(values) * 255).astype(np.uint8),
    )
np.savez_compressed(
    maps_dir / "scores_and_ids.npz",
    calib_class=DEV_CLASSES,
    val_class=val_classes,
    test_class=test_classes,
    val_iou=VAL_IOUS,
    calib_ids=calib_df.image_id.to_numpy(),
    val_ids=val_df.image_id.to_numpy(),
    test_ids=test_df.image_id.to_numpy(),
)
with zipfile.ZipFile(
    OUT / "level3_v3_maps.zip", "w", zipfile.ZIP_DEFLATED, allowZip64=True
) as archive:
    for path in maps_dir.iterdir():
        archive.write(path, path.name)

AUDIT = {
    "version": "level3_v3_clean_self_select",
    "seed": SEED,
    "winner_source_sha256": WINNER_SOURCE_SHA256,
    "device": torch.cuda.get_device_name(0),
    "selected": SELECTED["name"],
    "selection_rule": {
        "minimum_v3_gain": MIN_V3_GAIN,
        "validation_views": FINAL_VIEWS,
        "test_views": TEST_VIEWS,
        "snapshot": SELECTED_RESULT["state"],
    },
    "comparison": {
        "v2_selection_score": BASE_RESULT["selection_score"],
        "v3_selection_score": V3_RESULT["selection_score"],
        "v2_selection_fpr": BASE_RESULT["selection_fpr"],
        "v3_selection_fpr": V3_RESULT["selection_fpr"],
        "v2_calibration_score": BASE_RESULT["score"],
        "v3_calibration_score": V3_RESULT["score"],
    },
    "calibration": {
        "score": DEV_SCORE,
        "configuration": FINAL_CONFIG,
        "retrieval_coverage": float((np.asarray(calib_conf) >= 0.35).mean()),
    },
    "official_validation": {
        "roundtrip_mean_iou": VAL_SCORE,
        "positive_only_iou": VAL_POSITIVE,
        "gate_confusion_matrix": VAL_MATRIX,
        "zero_iou_fraction": float((VAL_IOUS == 0).mean()),
    },
    "final": {
        "epochs": len(final_history),
        "history": final_history,
        "submission_rows": len(submission),
        "nonempty": int(submission.polygon.str.strip().ne("").sum()),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "timm": timm.__version__,
        "opencv": cv2.__version__,
        "sklearn": sklearn.__version__,
    },
    "phase_times": PHASE_TIMES,
    "elapsed_hours": elapsed_hours(),
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
    "warnings": WARNINGS,
}
with open(OUT / "audit_level3_v3.json", "w", encoding="utf-8") as stream:
    json.dump(AUDIT, stream, indent=2, ensure_ascii=False)
print(json.dumps(AUDIT, indent=2, ensure_ascii=False))
print("READY", OUT / "submission.csv")


final {'epoch': 1, 'loss': 0.7176472299465393}
final {'epoch': 2, 'loss': 0.47908284746855834}
final {'epoch': 3, 'loss': 0.4097989479042504}
final {'epoch': 4, 'loss': 0.37451447401712595}
final {'epoch': 5, 'loss': 0.34420320743757793}
{
  "version": "level3_v3_clean_self_select",
  "seed": 20260727,
  "winner_source_sha256": "bd5dae5fff873e371e9b7f5bdd311d98ec9c7240dd65bc2628fdf93125b7d355",
  "device": "Tesla T4",
  "selected": "v3_enhanced",
  "selection_rule": {
    "minimum_v3_gain": 0.002,
    "validation_views": 2,
    "test_views": 2,
    "snapshot": "last"
  },
  "comparison": {
    "v2_selection_score": 0.9307188631971077,
    "v3_selection_score": 0.9392445263378243,
    "v2_selection_fpr": 0.0013568521031207597,
    "v3_selection_fpr": 0.0,
    "v2_calibration_score": 0.9314703856557379,
    "v3_calibration_score": 0.9385329343636843
  },
  "calibration": {
    "score": 0.9385329343636843,
    "configuration": {
      "alpha": 0.85,
      "gate_mode": "rank",
      "class